In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt


In [ ]:
results = pd.read_csv("./data/survey_results_public.csv", index_col="ResponseId")
resultSchema = pd.read_csv("./data/survey_results_schema.csv")

In [ ]:
results.head(5) 

In [ ]:
languages = "LanguageHaveWorkedWith" #Lenguaje
salary = "ConvertedCompYearly" #Salario Anual

#Renombramos las columnas.
results.rename(columns={languages: "Lenguajes", salary: "Salario"}, inplace = True ) #Remplaza el objeto en si. 

#Para poder calcular los salarios promedios Quitamos las filas cuya columna salario y/o lengauje sea NAN ya que no nos sirve.
results.dropna(subset=["Salario", "Lenguajes"], inplace=True)

#Nos quedaos con las columnas que nos interesa, es decir en Salario, Lenguaje y country
results = results.loc[:,["Country", "Salario", "Lenguajes"]].sort_values(by="Salario")

#Quitamos los extremos de Salario proque sabemos que no es real ni cobrar 1$ o 45M$
filtro = (results["Salario"] >= 10000) & (results["Salario"] <= 3e6)
results = results[filtro]



In [ ]:
#Debemos tener una lista con toda la relación de lenguajes distintos.
lg = results["Lenguajes"]
allLenguajes = []
for row in lg:
    for lang in row.split(";"):
        allLenguajes.append(lang)        
        
allLenguajes = list(set(allLenguajes)) #CGeneramos la lista de nuevo pero la orden set no coge los valores que ya existan.
allLenguajes.sort()
allLenguajes

In [ ]:
#Creamos un DataFrame donde el indice son los lenguajes y queremos una columna para el valor de salario medio.
#ponemos a 0 todas las celds de esa columna por eso indicamos el número de ceros que tiene que coincider con la longitud de lenguajes.
meanSalary = pd.DataFrame(data=np.zeros(len(allLenguajes)), index=allLenguajes, columns=["Salario"])

#Calculamos el Salario medio de cada lenguaje.
for lang in allLenguajes:
    try:
        mask = results["Lenguajes"].str.contains(lang)  
        average = results.loc[mask, "Salario"].mean()
        meanSalary.loc[lang, "Salario"] = average
    except Exception:
        print(lang)


In [ ]:
# Empezamos con el tema de mostrar valores en la pantalla. 
# Establecemos colores.
colors= [
    "#03071E",
    "#370617",
    "#6A040F",
    "#9D0208",
    "#D00000",
    "#DC2F02",
    "#E85D04",
    "#F48C06",
    "#FAA307",
    "#FFBA08"
]

# Establecemos las fuentes.  NO LO AMOS A USAR
fontdict = {
    'family' : "serif",
    "color" : "darkred",
    'weight' : 'normal',
    'size' : 16
}

In [24]:
#Esto lo hace para que el gráfico no se empotre en esta pantalla.
#%matplotlib qt5

meanSalary = meanSalary.sort_values(by="Salario") #o poniendo inplace=true para que no lo vuelva.
plt.style.use("seaborn-v0_8")
plt.barh(meanSalary.index, meanSalary["Salario"], color=colors)
 #pinta una línea vertica en la media de todos los salarios.  VA desde la posición 0 hasta el final de todo.
plt.vlines(meanSalary["Salario"].mean(), 0,len(meanSalary["Salario"]), colors="red", linestyles="dashed", label="Media Salarial" )
plt.title("Salario Medio Anual Lenguajes de Programación.")
plt.xlabel("Salario medio en dolares")
plt.ylabel("Lenguajes de programación")
plt.gca().xaxis.set_major_formatter('${x:1.0F}') # A cada valor del eje se le añade el símbolo del dolar.
plt.legend() #Para que aparezca la leyenda de la barra vertical.
plt.tight_layout()
plt.show()


